In [36]:
# 確保安裝 osmnx 及其相依套件，並強制更新以避免版本衝突
!pip install --upgrade osmnx folium networkx geopandas
import osmnx as ox
print(f"OSMNX 版本: {ox.__version__} 安裝成功！")

OSMNX 版本: 2.1.1 安裝成功！


In [44]:
import osmnx as ox
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import folium
from datetime import datetime
from IPython.display import display

# ==========================================
# 一、 路網拓撲與事故節點建置 (Graph Construction)
# ==========================================
ox.settings.timeout = 180
ox.settings.requests_timeout = 180
ox.settings.use_cache = True
ox.settings.overpass_url = "https://overpass-api.de/api/interpreter"

print("正在抓取高雄局部路網（半徑 800 公尺）...")
center_point = (22.624, 120.301)

try:
    G_raw = ox.graph_from_point(center_point, dist=800, network_type='drive')
    # OSMnx 2.x API 修正：直接呼叫 ox.project_graph
    G = ox.project_graph(G_raw)
    print("✅ 路網抓取成功！")
except Exception as e:
    print(f"❌ 抓取地圖失敗: {e}")
    raise e

nodes = list(G.nodes)
node_to_idx = {node: i for i, node in enumerate(nodes)}
N = len(nodes)

A = np.zeros((N, N))
for u, v, k, data in G.edges(keys=True, data=True):
    u_idx, v_idx = node_to_idx[u], node_to_idx[v]
    dist = data.get('length', 1.0)
    A[u_idx, v_idx] = np.exp(- (dist ** 2) / (500 ** 2))
    A[v_idx, u_idx] = A[u_idx, v_idx]

# ==========================================
# 二、 時空路況預測核心 (Spatio-Temporal Model)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LightSTGCN(nn.Module):
    def __init__(self, num_nodes, in_features, hidden_features, out_features, seq_len):
        super(LightSTGCN, self).__init__()
        self.A = torch.tensor(A, dtype=torch.float32).to(device)
        self.gcn_weights = nn.Linear(in_features, hidden_features)
        self.lstm = nn.LSTM(input_size=hidden_features, hidden_size=hidden_features, num_layers=1, batch_first=True)
        self.regressor = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        B, T, N, F = x.shape
        gcn_out = []
        for t in range(T):
            x_t = x[:, t, :, :]
            spatial_agg = torch.einsum('nn, bnf -> bnf', self.A, x_t)
            h_t = torch.relu(self.gcn_weights(spatial_agg))
            gcn_out.append(h_t.unsqueeze(1))
        gcn_out = torch.cat(gcn_out, dim=1)
        lstm_in = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        lstm_out, _ = self.lstm(lstm_in)
        pred = self.regressor(lstm_out[:, -1, :])
        return pred.reshape(B, N, -1)

model = LightSTGCN(num_nodes=N, in_features=1, hidden_features=16, out_features=3, seq_len=12).to(device)
model.eval()

dummy_input = torch.ones((1, 12, N, 1), dtype=torch.float32).to(device) * 40.0

start_node = nodes[0]
target_node = nodes[len(nodes) - 1]
path_static = nx.shortest_path(G, source=start_node, target=target_node, weight='length')
accident_node = path_static[len(path_static) // 2]
accident_idx = node_to_idx[accident_node]

with torch.no_grad():
    predicted_speeds = model(dummy_input).detach().cpu().numpy()[0]

# ==============================================================================
# 三、與四、 整合版：物理長度校正版 (Physical Distance Calibration)
# ==============================================================================

# 1. 同步與建立基準速度矩陣
calibrated_speeds = np.ones((N, 3)) * 55.0
calibrated_speeds[accident_idx, :] = predicted_speeds[accident_idx, :]
for neighbor in G.neighbors(accident_node):
    n_idx = node_to_idx[neighbor]
    calibrated_speeds[n_idx, :] = np.minimum(predicted_speeds[n_idx, :], 15.0)
calibrated_speeds[accident_idx, :] = 8.0

# ------------------------------------------
# 2. 計算尋路演算法與各路線預估時間 (ETA)
# ------------------------------------------
target_node = accident_node

# 方案 A：傳統最短距離導航
path_static = nx.shortest_path(G, source=start_node, target=target_node, weight='length')

# 方案 B：時空預測動態導航
for u, v, k, data in G.edges(keys=True, data=True):
    v_idx = node_to_idx[v]
    length = data.get('length', 1.0)
    pred_speed_kmh = max(calibrated_speeds[v_idx, 1], 5.0)
    pred_speed_ms = pred_speed_kmh / 3.6
    data['dynamic_cost'] = length / pred_speed_ms

path_dynamic = nx.shortest_path(G, source=start_node, target=target_node, weight='dynamic_cost')

# 利用物理總長度(Meters)進行實務時間分攤計算
dist_static = nx.path_weight(G, path_static, weight='length')
dist_dynamic = nx.path_weight(G, path_dynamic, weight='length')

# 消防實務科學公式
eta_static_pure = ((dist_static * 0.8) / (55.0 / 3.6)) + ((dist_static * 0.2) / (10.0 / 3.6))
eta_dynamic_pure = (dist_dynamic / (55.0 / 3.6))

dispatch_delay_sec = 60.0 # 1 分鐘派單延遲

total_sec_static = eta_static_pure + dispatch_delay_sec
total_sec_dynamic = eta_dynamic_pure + dispatch_delay_sec

# 計算分與秒
min_a, sec_a = divmod(int(total_sec_static), 60)
min_b, sec_b = divmod(int(total_sec_dynamic), 60)

time_saved_seconds = max(total_sec_static - total_sec_dynamic, 0.0)

# ------------------------------------------
# 3. 渲染互動式地圖與網頁面板 (Visualization)
# ------------------------------------------
G_wgs84 = ox.project_graph(G, to_crs='EPSG:4326')
m = folium.Map(location=center_point, zoom_start=15, tiles="CartoDB positron")

# 繪製全路網背景的壅塞熱度著色
for u, v, k, data in G_wgs84.edges(keys=True, data=True):
    v_idx = node_to_idx[v]
    speed = calibrated_speeds[v_idx, 1]

    if speed <= 10.0:
        edge_color, edge_weight, edge_opacity = "#D11A2A", 5, 0.9
    elif speed <= 20.0:
        edge_color, edge_weight, edge_opacity = "#FF4500", 4, 0.8
    else:
        edge_color, edge_weight, edge_opacity = "#2E8B57", 1.5, 0.35

    if 'geometry' in data:
        xs, ys = data['geometry'].xy
        points = list(zip(ys, xs))
    else:
        points = [(G_wgs84.nodes[u]['y'], G_wgs84.nodes[u]['x']), (G_wgs84.nodes[v]['y'], G_wgs84.nodes[v]['x'])]
    folium.PolyLine(points, color=edge_color, weight=edge_weight, opacity=edge_opacity).add_to(m)

folium.Marker([G_wgs84.nodes[start_node]['y'], G_wgs84.nodes[start_node]['x']], popup="消防分隊(起點)", icon=folium.Icon(color='green', icon='home')).add_to(m)
folium.Marker([G_wgs84.nodes[accident_node]['y'], G_wgs84.nodes[accident_node]['x']], popup="💥 車禍事故點", icon=folium.Icon(color='red', icon='exclamation-sign')).add_to(m)

route_static_coords = [(G_wgs84.nodes[n]['y'], G_wgs84.nodes[n]['x']) for n in path_static]
folium.PolyLine(route_static_coords, color="#222222", weight=7, opacity=0.8).add_to(m)

route_dynamic_coords = [(G_wgs84.nodes[n]['y'], G_wgs84.nodes[n]['x']) for n in path_dynamic]
folium.PolyLine(route_dynamic_coords, color="#00FF00", weight=4, opacity=1.0).add_to(m)

dashboard_html = f'''
<div style="position: fixed; top: 20px; left: 70px; width: 290px; background: white; padding: 15px; border-radius: 10px; box-shadow: 3px 3px 6px rgba(0,0,0,0.2); z-index:9999; font-family: 'Microsoft JhengHei', sans-serif;">
    <h4 style="margin-top:0;"> 119 救護路線分析</h4>
    方案 A (傳統最短距離): {min_a}分{sec_a}秒<br>
    方案 B (STGCN 動態導航): {min_b}分{sec_b}秒<br>
    <b>⚡️ 節省時間: {int(time_saved_seconds)} 秒</b>
</div>
'''
m.get_root().html.add_child(folium.Element(dashboard_html))
display(m)

正在抓取高雄局部路網（半徑 800 公尺）...
✅ 路網抓取成功！
